# Boosting


### What Is Boosting?

<span style="color:#1f77b4">**Boosting**</span> is an <span style="color:#1f77b4">**ensemble**</span> technique that combines many <span style="color:#ff7f0e">**weak predictive models**</span>—typically shallow decision trees—into one strong learner.

By fitting models <span style="color:#ff7f0e">**sequentially**</span> and letting each new model focus on the <span style="color:#ff7f0e">**errors**</span> of the previous ones, Boosting steadily <span style="color:#2ca02c">**reduces bias**</span> and improves overall accuracy.

---

### How It Works

Conceptual algorithm (<span style="color:#1f77b4">**Adaptive Boosting**</span>, or AdaBoost):

1. Start with <span style="color:#ff7f0e">**uniform observation weights**</span>.
2. Train a **stump**, i.e., a one-level tree (refered to as a weak learner in adaBoost).
3. <span style="color:#ff7f0e">**Up-weight**</span> the rows the stump misclassified.
4. Train the next stump on the re-weighted data.
5. <span style="color:#1f77b4">**Combine**</span> trees in a <span style="color:#ff7f0e">**weighted vote**</span> (later trees usually get higher weight).
6. Repeat for $M$ rounds.

---

### Why it Works

* Each tree focuses on what its predecessors missed ⇒ <span style="color:#2ca02c">**reduces bias**</span>.
* Shallow trees + <span style="color:#ff7f0e">**small learning rate**</span> keep variance in check.

---

### Popular Boosting methods

| Algorithm                         | Core Idea                                                                                             |
| --------------------------------- | ----------------------------------------------------------------------------------------------------- |
| **AdaBoost**                      | Adjust sample weights via exponential loss; emphasize hard cases.                                     |
| **Gradient Boosting**             | Fit each learner to the *negative gradient* of a differentiable loss (e.g., squared, log-loss).       |
| **XGBoost / LightGBM / CatBoost** | Engineering-heavy gradient boosting variants with tree pruning, regularization, and GPU acceleration. |

---

### Pros and Cons

✅ Often delivers *state-of-the-art* accuracy out-of-the-box.  
✅ **Reduces bias** by turning weak learners into a strong composite.  
✅ Handles **mixed feature types** and missing values (tree-based versions).  
✅ Offers many regularization knobs (learning rate, subsampling) to fight overfitting.  

❌ **Sequential training** means poor parallelism and longer training times.  
❌ Can *overfit* if the ensemble grows too deep or learning rate is too high.  
❌ Model is less interpretable than a single tree (though feature importance helps).  

### Key Hyper-parameters

| Symbol                         | Meaning             | Tips                                          |
| ------------------------------ | ------------------- | --------------------------------------------- |
| `n_estimators`                 | rounds $M$          | 100–500 common                                |
| `learning_rate`                | shrinkage per round | 0.01–0.1 (smaller = safer, needs more rounds) |
| `max_depth` / `max_leaf_nodes` | size of each tree   | 2–6 for tabular                               |



> *Boosting “learns from its mistakes,” layering many simple models into a powerful predictor that often outperforms standalone learners—provided you tune it with care.*  --- Words of wisdom by ChatGPT 3o (2025)

> *Boosting grows strong by returning to what it got wrong—turning many small corrections into one powerful prediction. But chase every error too eagerly, and noise becomes the teacher.* --- Words of wisdom by GPT-5.6 Pro (2026)

> *Boosting is an apprenticeship: every new learner studies exactly where the last one failed --- keep each apprentice humble and each step small, or the ensemble starts memorizing the very noise it was meant to overcome.* --- Words of wisdom by Claude Fable 5 


We will examine the same simulated example. 

In [1]:
# --- Synthetic data: three classes separated by two straight lines -------
# (Same data-generating process as the bagging notebook; the "truth" is
#  known, so we can see exactly how well each method recovers it.)
import numpy as np

rng = np.random.default_rng(0)   # fixed seed: everyone gets the same data
# Examples with more data points:
sample_size = 200
X = rng.uniform(0.1, 0.9, size=(sample_size, 2))   # 200 points, 2 features

# Assign labels by region (no noise -- the boundaries ARE the truth):
y = np.zeros(sample_size, dtype=int)
mask1 = X[:, 0] + X[:, 1] > 1.1            # above the line x1 + x2 = 1.1  -> class 1
mask2 = (~mask1) & (X[:, 0] - X[:, 1] > 0.3)  # below it but right of x1 - x2 = 0.3 -> class 0
y[mask1] = 1
y[mask2] = 0
y[~(mask1 | mask2)] = 2                    # everything else -> class 2


In [2]:
# --- numerical & plotting -------------------------------------------------
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap   # only if you still need to define label_cmap
# --- machine learning -----------------------------------------------------
import sklearn                                # to check sklearn.__version__
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# --- widgets / interactivity ---------------------------------------------
import ipywidgets as widgets                  # IntSlider
from ipywidgets import interact               # high-level helper

label_cmap = ListedColormap(["#1f77b4", "#ff7f0e", "#2ca02c"])

# ----- helper: fit & plot an AdaBoost ensemble ------------------------------
# The six algorithm steps from "How It Works" all happen inside
# AdaBoostClassifier.fit():
#   Step 1 (uniform weights):  every row starts with weight 1/n.
#   Step 2 (weak learner):     one DecisionTreeClassifier(max_depth=...) per
#                              round; max_depth=1 is the classic stump.
#   Step 3 (up-weight errors): rows the current tree misclassifies get
#                              heavier weights.
#   Step 4 (refit):            the next tree trains on the re-weighted data,
#                              so it concentrates on what was missed.
#   Step 5 (weighted vote):    .predict() combines all trees, each with its
#                              own weighted say.
#   Step 6 (repeat M rounds):  n_estimators is M; learning_rate shrinks each
#                              round's contribution (smaller = safer).
def plot_adaboost(max_depth=1, n_estimators=100):
    # Build AdaBoost with the right keyword for your scikit-learn version
    ada_kwargs = dict(
        n_estimators=n_estimators,    # Step 6: M boosting rounds
        learning_rate=0.5,            # Step 6: shrinkage per round
        random_state=42,
    )
    if sklearn.__version__ >= "1.4":
        ada_kwargs["estimator"] = DecisionTreeClassifier(max_depth=max_depth)      # Step 2: the weak learner
    else:
        ada_kwargs["base_estimator"] = DecisionTreeClassifier(max_depth=max_depth) # pre-1.4 API

    ada = AdaBoostClassifier(**ada_kwargs)
    ada.fit(X, y)                     # Steps 1-6 (the sequential loop) run here

    # Visualization: ask the fitted ensemble to classify every point on a fine
    # grid, so the colored regions show the ensemble's decision rule (Step 5's
    # weighted vote, evaluated everywhere).
    # Create a fine mesh over the feature space
    x_min, x_max = X[:, 0].min() - 0.05, X[:, 0].max() + 0.05
    y_min, y_max = X[:, 1].min() - 0.05, X[:, 1].max() + 0.05
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )
    Z = ada.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Plot
    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap=label_cmap)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=label_cmap,
                edgecolors="k", s=80)
    plt.title(f"AdaBoost: max_depth = {max_depth}, "
              f"n_estimators = {n_estimators}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.tight_layout()
    plt.show()

# What each slider teaches:
#   max_depth:     1 = the classic stump (a genuinely weak learner); deeper
#                  base trees are no longer "weak"
#   n_estimators:  more rounds = more sequential corrections; unlike bagging,
#                  more rounds CAN overfit
# 3. ── interactive sliders (depth 1-4, trees 100-500) ──────────────────
interact(
    plot_adaboost,
    max_depth=widgets.IntSlider(min=1, max=4, step=1, value=1,
                                description="max_depth"),
    n_estimators=widgets.IntSlider(min=100, max=500, step=100, value=100,
                                   description="n_estimators")
);


interactive(children=(IntSlider(value=1, description='max_depth', max=4, min=1), IntSlider(value=100, descript…